# Installation of libraries to run Spark

In [1]:
!pip install pyspark
!pip install findspark


In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Create a Spark session
spark = SparkSession.builder\
    .master("local[*]")\
    .appName("Spark_cleaning")\
    .getOrCreate()


In [3]:
df = (spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("/content/sample_data/dataset.csv")
)
# Display schema and data
df.printSchema()
df.show()


root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)

+--------------------+--------------------+------------------+--------------------+-------+----------------+----------------+------------+------+--------+------------+--------------+-------+---------------------+
|      transaction_id|         customer_id|     customer_name|               email|country|            city|product_category|product_name| price|quantity|total_amount|payment_method

In [5]:
df.describe()

DataFrame[summary: string, transaction_id: string, customer_id: string, customer_name: string, email: string, country: string, city: string, product_category: string, product_name: string, price: string, quantity: string, total_amount: string, payment_method: string, status: string]

In [6]:
from pyspark.sql.functions import col, sum, when, count

In [7]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+
|transaction_id|customer_id|customer_name|email|country|city|product_category|product_name|price|quantity|total_amount|payment_method|status|transaction_timestamp|
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+
|             0|          0|            0|    0|      0|   0|               0|           0|    0|       0|           0|             0|     0|                    0|
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+



I dont have any null values, So I have to check for duplicates

In [8]:
df.select("transaction_id").show()

+--------------------+
|      transaction_id|
+--------------------+
|624ef8ad-95cb-499...|
|976f73bf-2521-4ce...|
|d44b5663-db35-4e8...|
|c486435d-b1e0-4e5...|
|b50eb77f-c517-46e...|
|98444a89-ce62-4ee...|
|5a6c8dbf-944a-472...|
|ce8f675e-0efd-4ed...|
|8864891b-45b9-44a...|
|c6e15f74-0882-43d...|
|7b2d3f61-ffc7-4d2...|
|050a447a-819c-451...|
|10b2b8f4-163b-47e...|
|316d36f0-2b4e-4d1...|
|0700ce62-1280-407...|
|dea6f523-0dc7-463...|
|b274815a-93b1-497...|
|8a9f92a2-e843-4a6...|
|77f67d5f-cbbc-41f...|
|b7224649-fa0c-4f5...|
+--------------------+
only showing top 20 rows


In [9]:
#find the duplicate rows
duplicate_rows_df = df.groupBy(df.columns).agg(
    count("*").alias("count")
).filter(
    col("count") > 1
)

duplicate_rows_df.show()

+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+-----+
|transaction_id|customer_id|customer_name|email|country|city|product_category|product_name|price|quantity|total_amount|payment_method|status|transaction_timestamp|count|
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+-----+
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+-----+



In [10]:
negative_prices = df.filter(col('price') < 0)
negative_quantities = df.filter(col('quantity') < 0)

In [12]:
negative_prices.show()

+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+
|transaction_id|customer_id|customer_name|email|country|city|product_category|product_name|price|quantity|total_amount|payment_method|status|transaction_timestamp|
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+



In [13]:
import pyspark.sql.functions as f
from pyspark.sql.functions import mean, stddev , col

In [16]:
from pyspark.sql.functions import mean, stddev, col

stats = df.select(
    mean(col('price')).alias('mean'),
    stddev(col('price')).alias('stddev')
).collect()

mean_val = stats[0]['mean']
stddev_val = stats[0]['stddev']


In [18]:
from pyspark.sql.functions import col, abs

df_with_z = df.withColumn(
    'z_score',
    (col('price') - mean_val) / stddev_val
)

df_outliers = df_with_z.filter(abs(col('z_score')) > 3)


In [19]:
df_outliers.show()

+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+-------+
|transaction_id|customer_id|customer_name|email|country|city|product_category|product_name|price|quantity|total_amount|payment_method|status|transaction_timestamp|z_score|
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+-------+
+--------------+-----------+-------------+-----+-------+----+----------------+------------+-----+--------+------------+--------------+------+---------------------+-------+

